# AgroVisión — Entrenamiento en Kaggle con TPU v5e-8

Notebook optimizado para Kaggle con TPU v5e-8 (8 núcleos).

> **Antes de ejecutar:** En Kaggle ve a `Settings → Accelerator → TPU v5e-8`  
> Si ya tenías el notebook abierto sin TPU, haz **Factory Reset** (Settings → Factory Reset)
> antes de seleccionar la TPU para que los drivers queden bien inicializados.

---

### Qué produce

Cinco archivos indivisibles que la app instala como una unidad:

| archivo | qué es |
|---|---|
| `model.tflite` | el modelo cuantizado, ~5 MB, con **dos salidas**: logits y embedding |
| `labels.json` | orden canónico de las clases y su vínculo con el catálogo |
| `calibration.json` | temperatura, umbrales de las tres compuertas, centroides y matriz de precisión |
| `metrics.json` | F1 por clase, matriz de confusión, paridad, trazabilidad del dataset |
| `signature.bin` | firma Ed25519 sobre el conjunto — sin ella ningún teléfono lo instala |

### Datos

JMuBEN + JMuBEN2 · 58 555 imágenes de café arábica tomadas en campo en
Kirinyaga, Kenia, con acompañamiento de un patólogo · CC BY 4.0.

> Jepkoech, J.; Kenduiywo, B.; Mugo, D.; Chebet, E. (2021). *Arabica coffee leaf
> images dataset for coffee leaf disease detection and classification*.
> Data in Brief 36, 107142.

## 1 · Comprobar la TPU

Esta celda inicializa la TPU v5e-8. Debe mostrar `REPLICAS: 8`.  
Si falla, ve a **Settings → Factory Reset**, luego **Settings → Accelerator → TPU v5e-8** y vuelve a ejecutar.

In [ ]:
import os
import tensorflow as tf

# ── Inicialización TPU v5e-8 en Kaggle ───────────────────────────────────────
# La TPU v5e es una TPU VM: el chip está en la misma máquina.
# Por eso se conecta con tpu='local', no con una dirección de red.
try:
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu='local')
    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    STRATEGY = tf.distribute.TPUStrategy(resolver)
    ACELERADOR = 'TPU'
    print(f'✓ TPU v5e-8 inicializada correctamente')
    print(f'  REPLICAS: {STRATEGY.num_replicas_in_sync}')
    print(f'  Dispositivos lógicos: {tf.config.list_logical_devices("TPU")}')
except Exception as e:
    raise RuntimeError(
        f'No se pudo conectar a la TPU: {e}\n\n'
        'Solución: Settings → Factory Reset, luego Settings → Accelerator → TPU v5e-8'
    )


## 2 · Traer el pipeline

Se clona el repositorio con el código del entrenamiento.


In [ ]:
import pathlib, shutil

REPO = 'https://github.com/ScorpkID/coffeApp-mll.git'

destino = pathlib.Path('/kaggle/working/agrovision-ml')
shutil.rmtree(destino, ignore_errors=True)
!git clone --depth 1 {REPO} {destino}

RAIZ = destino
assert (RAIZ / 'pyproject.toml').exists(), f'No encuentro pyproject.toml en {RAIZ}'
print('✓ Repositorio clonado en', RAIZ)


## 3 · Instalar dependencias

Instala el pipeline y verifica que Keras 3 esté disponible.


In [ ]:
# ── Instalación sin tocar TensorFlow ─────────────────────────────────────────
# En Kaggle, TF viene pre-instalado con libtpu (soporte TPU).
# Si pip reinstala TF desde PyPI, lo reemplaza por uno SIN kernels de TPU
# y la TPU deja de funcionar. Por eso instalamos SOLO el resto del pipeline.
%pip install -q --no-deps -e {RAIZ}
%pip install -q numpy pillow scikit-learn typer rich pyyaml cryptography requests tqdm 2>&1 | tail -3

from tensorflow import keras

print('tensorflow', tf.__version__, '· keras', keras.__version__)
print('Acelerador activo:', ACELERADOR)
print(f'Batch global: {STRATEGY.num_replicas_in_sync} núcleos × 32 = {STRATEGY.num_replicas_in_sync * 32} imágenes/paso')

assert keras.__version__.startswith('3'), (
    f'Se esperaba Keras 3, hay {keras.__version__}.'
)
print('✓ Todo listo para entrenar')


## 4 · Claves y rutas de datos

Los ZIPs del dataset están en `/kaggle/working/agrovision-raw/`.  
La clave de firma se genera en `/kaggle/working/agrovision/secretos/` — **descárgala al terminar**,  
si la pierdes no podrás firmar el próximo modelo y ningún teléfono lo aceptará.


In [ ]:
import pathlib

# ── Clave de firma ────────────────────────────────────────────────────────────
BASE     = pathlib.Path('/kaggle/working/agrovision')
SECRETOS = BASE / 'secretos'
SECRETOS.mkdir(parents=True, exist_ok=True)
CLAVE    = SECRETOS / 'model_signing_private.pem'

if CLAVE.exists():
    print('✓ Clave ya existe — se reutiliza')
else:
    !agrovision-ml keys --out {SECRETOS}

print('Clave pública (cópiala a gradle.properties):')
print((SECRETOS / 'model_signing_public.b64').read_text())

# ── Rutas de datos ────────────────────────────────────────────────────────────
# Los ZIPs subidos a Kaggle quedan en /kaggle/working/agrovision-raw/
CACHE = pathlib.Path('/kaggle/working/agrovision-raw')
DATOS = pathlib.Path('/kaggle/working/datos')
DATOS.mkdir(exist_ok=True)

# Enlace simbólico para que el pipeline encuentre los ZIPs donde los espera
enlace = DATOS / 'raw'
if enlace.exists() or enlace.is_symlink():
    enlace.unlink()
enlace.symlink_to(CACHE)

print(f'✓ ZIPs en: {CACHE}')
print(f'  Archivos encontrados: {list(CACHE.glob("*.zip"))}')


## 5 · Extraer el dataset

Los ZIPs ya están en Kaggle — esta celda solo los descomprime.  
Si ya están descomprimidos de una corrida anterior, pasa de largo.


In [ ]:
# Los ZIPs ya están en /kaggle/working/agrovision-raw/, no hay nada que descargar.
# El pipeline los encuentra y los descomprime a DATOS/images/.
!agrovision-ml download --config {RAIZ}/configs/coffee_v1.yaml --data {DATOS}


## 5.5 · Limpieza de imágenes corruptas

Verifica todas las imágenes extraídas y elimina las dañadas antes de entrenar.  
Un archivo corrupto en medio del entrenamiento lo detiene completamente.


In [ ]:
from PIL import Image
from tqdm.notebook import tqdm
from pathlib import Path

images_dir = DATOS / 'images'
corruptas  = 0
revisadas  = 0

if images_dir.exists():
    todas = list(images_dir.rglob('*'))
    imagenes = [p for p in todas if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    for ruta in tqdm(imagenes, desc='Verificando imágenes'):
        revisadas += 1
        try:
            with Image.open(ruta) as img:
                img.verify()
        except Exception:
            print(f'  ✗ Corrupta, eliminada: {ruta.name}')
            ruta.unlink()
            corruptas += 1
    print(f'\n✓ {revisadas} imágenes revisadas — {corruptas} corruptas eliminadas')
else:
    print('⚠️ Carpeta de imágenes no encontrada. Ejecuta la celda 5 primero.')


## 6 · Entrenar

Con TPU v5e-8 (8 núcleos) y batch 256 esto debería tomar unos **5–10 minutos** en total.  

El pipeline completo:
```
reparto → etapa 1 (cabeza) → etapa 2 (ajuste fino) → calibración →
evaluación → exportación INT8 → paridad → aceptación → firma
```
Si el modelo no alcanza los mínimos de la receta, **se detiene antes de firmar**.


In [ ]:
ARTEFACTOS = pathlib.Path('/kaggle/working/artefactos')

!agrovision-ml train \
    --config {RAIZ}/configs/coffee_v1.yaml \
    --data   {DATOS} \
    --out    {ARTEFACTOS} \
    --key    {CLAVE}


## 7 · Revisar el informe

Lo que hay que mirar antes de publicar nada.


In [ ]:
import json

version  = next(ARTEFACTOS.glob('v*'))
informe  = json.loads((version / 'metrics.json').read_text())

print(f"F1 macro    {informe['evaluation']['macro_f1']}")
print(f"exactitud   {informe['evaluation']['accuracy']}")
print(f"temperatura {informe['calibration']['temperature']}")
print(f"ECE         {informe['calibration']['ece_before']} → {informe['calibration']['ece_after']}")
print(f"tamaño      {informe['export']['size_bytes'] / 1048576:.2f} MB")
print()
print('Por clase:')
for fila in informe['evaluation']['per_class']:
    print(f"  {fila['class_id']:30} F1 {fila['f1']:.4f}  ({fila['support']:,} imágenes)")
print()
print('De cada 100 fotos, la app respondería:')
for clave, valor in informe['evaluation']['gate_distribution'].items():
    print(f"  {clave:16} {valor * 100:5.1f}")


### Matriz de confusión

Dónde se confunde el modelo importa más que cuánto.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

matriz     = np.array(informe['evaluation']['confusion_matrix'])
nombres    = [c['class_id'] for c in informe['evaluation']['per_class']]
normalizada = matriz / np.maximum(matriz.sum(axis=1, keepdims=True), 1)

figura, eje = plt.subplots(figsize=(7, 6))
eje.imshow(normalizada, cmap='YlOrBr', vmin=0, vmax=1)
eje.set_xticks(range(len(nombres)), nombres, rotation=45, ha='right')
eje.set_yticks(range(len(nombres)), nombres)
eje.set_xlabel('predicho'); eje.set_ylabel('real')
for i in range(len(nombres)):
    for j in range(len(nombres)):
        eje.text(j, i, f'{normalizada[i, j]:.2f}', ha='center', va='center',
                 color='white' if normalizada[i, j] > 0.5 else 'black', fontsize=9)
plt.tight_layout(); plt.show()


## 8 · Descargar los artefactos

Los archivos generados quedan en `/kaggle/working/`.  
Ve al panel derecho de Kaggle → **Output** para descargarlos.

> ⚠️ **Descarga también la clave privada** (`model_signing_private.pem`) y guárdala en un lugar seguro.
> Si la pierdes, no podrás firmar el próximo modelo y los teléfonos existentes lo rechazarán.


In [ ]:
import shutil

# Elimina los pesos intermedios (cientos de MB) — no viajan con el modelo
shutil.rmtree(version / 'trabajo', ignore_errors=True)

# Empaqueta los 5 artefactos en un zip para descarga fácil
paquete = shutil.make_archive('/kaggle/working/agrovision-modelo', 'zip', version)
print(f'✓ Paquete listo: {paquete}')
print(f'  Tamaño: {pathlib.Path(paquete).stat().st_size / 1048576:.1f} MB')
print()
print('Archivos disponibles en Output:')
print('  • agrovision-modelo.zip  → modelo + calibración + métricas + firma')
print(f'  • agrovision/secretos/model_signing_private.pem  → GUÁRDALA')
